# QLoRA fine-tuning: ticket-router (Qwen2.5-1.5B-Instruct)

Fine-tunes the open-weights model set in `train/config.yaml` (`base_model`, default
`Qwen/Qwen2.5-1.5B-Instruct`) on the ticket-routing SFT data from step-06, so it can be
self-hosted, quantized, and served in Docker in later steps (09-12). **Not** a hosted-API
fine-tune -- see the project's `README.md` "Next steps" section for why.

**Runs on Google Colab only (free T4 GPU runtime). No training happens on the local machine.**

## Expected runtime (free T4, approximate -- varies with Colab's GPU allocation/throttling)
- Smoke test (~100 steps): **~5-10 minutes**
- Full run (3 epochs, ~2,450 steps at effective batch size 16): **~45-90 minutes**

## Expected VRAM (free T4 has 16 GB total)
- **~6-10 GB** during training (4-bit NF4 base model + LoRA adapters + activations with
  gradient checkpointing + paged 8-bit optimizer states) -- comfortable headroom on a T4.

## Cell order -- run top to bottom, in this exact order
1. Check GPU (must show a T4 or better -- stop if none)
2. Install dependencies
3. Get the repo (git clone for code + Google Drive/upload for the gitignored data files)
4. Load `train/config.yaml`
5. Load tokenizer + SFT datasets
6. **Verify completion-only masking** (prints one full example's per-token labels)
7. **Verify padding is dynamic per-batch, not fixed to `max_seq_len`**
8. Load base model in 4-bit (QLoRA) + attach LoRA
9. Build the trainer factory from `train/config.yaml` (nothing hardcoded here)
10. **Smoke test** (~100 steps) -- loss curve + 5 generated JSON outputs
11. --- STOP AND CONFIRM before continuing (read the smoke-test results first) ---
12. **Full training run**
13. Save the adapter + push it to the Hugging Face Hub
14. **Post-train preview**: generate on 20 val examples, print alongside gold labels

> **A note on what was and wasn't verified before this notebook was written**: the
> completion-only-masking mechanism (cell 6) and the dynamic-padding behavior (cell 7) were
> verified by hand, token-by-token, against Qwen2.5's real tokenizer before this notebook was
> written -- see `PROGRESS.md`'s step-07 entry for the full worked example. The end-to-end
> QLoRA training loop itself (bitsandbytes 4-bit loading, the actual GPU training step) could
> **not** be executed locally (no GPU available, and a local disk-space constraint blocked even
> a CPU-only `trl`/`torch` install for further testing) -- it's written to the current standard
> `transformers`/`trl`/`peft` API, but you should expect to debug minor version-API mismatches
> on first run. Please report back what you see so this can be corrected if needed.

## 1. Check GPU

In [ ]:
!nvidia-smi

# Expect a Tesla T4 in the table above. If this errors or shows no GPU:
# Runtime menu -> Change runtime type -> T4 GPU -> Save, then re-run from the top.

## 2. Install dependencies

Training-specific packages, installed only inside this Colab runtime -- not part of the
project's own `requirements.txt` (that file is for the local data/harness pipeline, which
never needs a GPU training stack).

In [ ]:
!pip install -q -U "transformers>=4.46" "trl>=0.12" "peft>=0.13" "accelerate>=1.0" "bitsandbytes>=0.44" datasets pyyaml huggingface_hub

## 3. Get the repo

`data/processed/` (the parquet splits and the SFT `.jsonl` files from step-06) is **gitignored** -- a plain `git clone` will get you the notebook, `train/config.yaml`, and the `src/` package, but **not** the actual training data. Pick ONE of the two options below.

**Option A (recommended if you've pushed this repo to GitHub):** clone for the code, then upload just the two `.jsonl` files (small enough for a direct Colab upload).

**Option B (no GitHub remote needed):** sync your entire local repo folder -- gitignored files included -- to Google Drive, then mount Drive here.

In [ ]:
import os

# --- Option A: git clone (code) + direct upload (data) ---------------------------------
# !git clone https://github.com/<your-username>/ticket-router.git
# REPO_ROOT = "/content/ticket-router"
# Then use the Colab file-browser (left sidebar) to upload train.jsonl/val.jsonl into
# REPO_ROOT + "/data/processed/sft/", or use the Files > Upload widget.

# --- Option B: Google Drive (default -- edit DRIVE_REPO_PATH to match your setup) ------
from google.colab import drive
drive.mount('/content/drive')
DRIVE_REPO_PATH = "/content/drive/MyDrive/ticket-router"  # <-- EDIT to where you placed the repo
REPO_ROOT = DRIVE_REPO_PATH

assert os.path.isdir(REPO_ROOT), f"{REPO_ROOT} not found -- upload/sync the repo there, or use Option A above."
assert os.path.exists(f"{REPO_ROOT}/train/config.yaml"), "train/config.yaml not found under REPO_ROOT."
assert os.path.exists(f"{REPO_ROOT}/data/processed/sft/train.jsonl"), (
    "data/processed/sft/train.jsonl not found. This directory is gitignored, so a plain "
    "git clone alone will NOT include it -- run src/ticket_router/make_sft_dataset.py "
    "locally first, then upload/sync data/processed/sft/*.jsonl separately."
)
print("REPO_ROOT OK:", REPO_ROOT)

## 4. Load `train/config.yaml`

Every hyperparameter used below comes from this file -- nothing is hardcoded in this notebook.

In [ ]:
import sys, yaml

sys.path.insert(0, f"{REPO_ROOT}/src")  # so `from ticket_router...` works later (schema reuse)

with open(f"{REPO_ROOT}/train/config.yaml") as f:
    config = yaml.safe_load(f)

print(yaml.dump(config, sort_keys=False))

## 5. Load tokenizer + SFT datasets

In [ ]:
from transformers import AutoTokenizer
from datasets import load_dataset

tokenizer = AutoTokenizer.from_pretrained(config["base_model"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

train_path = f"{REPO_ROOT}/{config['train_file']}"
val_path = f"{REPO_ROOT}/{config['val_file']}"

raw_datasets = load_dataset("json", data_files={"train": train_path, "val": val_path})
print(raw_datasets)
print("\nOne raw example's messages:")
print(raw_datasets["train"][0]["messages"])

## 6. Verify completion-only masking

The loss must be computed **only** on the assistant's JSON response -- system and user tokens
are masked out (label = `-100`, PyTorch's cross-entropy `ignore_index`, so they contribute
nothing to the loss or its gradient). Otherwise most of the model's fine-tuning capacity goes
into learning to *write* support tickets (reproducing the user turn) instead of learning to
*classify* them (producing the assistant turn).

**Mechanism: `trl.DataCollatorForCompletionOnlyLM`**, given the exact token sequence for
Qwen's chat-template assistant marker (`response_template` from `train/config.yaml`,
`"<|im_start|>assistant\n"`).

Verified by hand before this notebook was written (see `PROGRESS.md`'s step-07 entry for the
full worked example): for Qwen2.5's tokenizer, `<|im_start|>` and `<|im_end|>` are **protected
special tokens**, so `response_template` tokenizes identically whether it's standalone or
embedded in a full formatted sequence -- no silent boundary mismatch (the classic failure mode
with this collator on tokenizers that don't protect these markers). The cell below reproduces
that exact check live, using the actual `trl` collator, and prints one full example's
per-token labels so you can see the prompt tokens are ignored.

In [ ]:
from trl import DataCollatorForCompletionOnlyLM

response_template = config["response_template"]
collator = DataCollatorForCompletionOnlyLM(response_template=response_template, tokenizer=tokenizer)

def tokenize_example(example):
    ids = tokenizer.apply_chat_template(
        example["messages"], tokenize=True, add_generation_prompt=False,
        truncation=True, max_length=config["max_seq_len"],
    )
    return {"input_ids": ids}

tokenized_train = raw_datasets["train"].map(tokenize_example, remove_columns=raw_datasets["train"].column_names)
tokenized_val = raw_datasets["val"].map(tokenize_example, remove_columns=raw_datasets["val"].column_names)

# Sanity-check the collator on ONE real example.
example = tokenized_train[0]
batch = collator([{"input_ids": example["input_ids"]}])
input_ids = batch["input_ids"][0].tolist()
labels = batch["labels"][0].tolist()

n_masked = sum(1 for l in labels if l == -100)
n_loss = len(labels) - n_masked
print(f"Example 0: {len(labels)} tokens total, {n_masked} masked (-100), {n_loss} contribute to loss.\n")

print(f"{'idx':>4} {'token_id':>8}  {'label':>8}  token_text")
for i, (tid, lab) in enumerate(zip(input_ids, labels)):
    tok_str = tokenizer.convert_ids_to_tokens([tid])[0]
    marker = "MASKED" if lab == -100 else "LOSS  "
    print(f"{i:>4} {tid:>8}  {str(lab):>8}  [{marker}] {tok_str!r}")

loss_token_ids = [tid for tid, lab in zip(input_ids, labels) if lab != -100]
print("\nDecoded loss-only tokens (should be exactly the assistant JSON + <|im_end|>):")
print(repr(tokenizer.decode(loss_token_ids)))

assert n_loss > 0, (
    "No unmasked tokens found -- response_template did not match this example. "
    "STOP: do not train until this is fixed, or every example's loss would be masked entirely."
)

## 7. Confirm padding: dynamic per-batch, not fixed to `max_seq_len`

`max_seq_len` in `train/config.yaml` is a **truncation cap**, not a padding target (this was
left open in step-06). Verified by hand before this notebook was written: `tokenizer.pad()`
(which `DataCollatorForCompletionOnlyLM` uses internally) pads each batch to **that batch's
own longest sequence**, not to a fixed length. The cell below reproduces that check against
the actual collator built above, with two differently-sized batches.

In [ ]:
batch_a = [tokenized_train[i] for i in range(4)]
batch_b = [tokenized_train[i] for i in range(4, 4 + 8)]  # a differently-sized batch

padded_a = collator([{"input_ids": ex["input_ids"]} for ex in batch_a])
padded_b = collator([{"input_ids": ex["input_ids"]} for ex in batch_b])

print("batch A native lengths:", [len(ex["input_ids"]) for ex in batch_a])
print("batch A padded shape:  ", tuple(padded_a["input_ids"].shape))
print()
print("batch B native lengths:", [len(ex["input_ids"]) for ex in batch_b])
print("batch B padded shape:  ", tuple(padded_b["input_ids"].shape))
print()
print(f"max_seq_len from config: {config['max_seq_len']}")
print("If padding were fixed to max_seq_len, BOTH batches would show that same second dimension.")
print("Instead each batch is padded independently to its own longest row -- dynamic per-batch padding, confirmed.")

## 8. Load base model in 4-bit (QLoRA) + attach LoRA

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=config["quantization"]["load_in_4bit"],
    bnb_4bit_quant_type=config["quantization"]["bnb_4bit_quant_type"],
    bnb_4bit_compute_dtype=getattr(torch, config["quantization"]["bnb_4bit_compute_dtype"]),
    bnb_4bit_use_double_quant=config["quantization"]["bnb_4bit_use_double_quant"],
)

model = AutoModelForCausalLM.from_pretrained(
    config["base_model"],
    quantization_config=bnb_config,
    device_map="auto",
)
model = prepare_model_for_kbit_training(
    model, use_gradient_checkpointing=config["gradient_checkpointing"]
)

lora_config = LoraConfig(
    r=config["lora"]["r"],
    lora_alpha=config["lora"]["lora_alpha"],
    lora_dropout=config["lora"]["lora_dropout"],
    bias=config["lora"]["bias"],
    task_type=config["lora"]["task_type"],
    target_modules=config["lora"]["target_modules"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 9. Trainer factory

One function, reused for both the smoke test and the full run, so the two runs differ only in
`max_steps`/`num_train_epochs` -- every other hyperparameter comes from `train/config.yaml`.

In [ ]:
from trl import SFTConfig, SFTTrainer

def build_trainer(output_dir, max_steps=None, num_train_epochs=None, eval_steps=None, logging_steps=None):
    sft_config = SFTConfig(
        output_dir=output_dir,
        per_device_train_batch_size=config["per_device_train_batch_size"],
        per_device_eval_batch_size=config["per_device_eval_batch_size"],
        gradient_accumulation_steps=config["gradient_accumulation_steps"],
        learning_rate=float(config["learning_rate"]),
        lr_scheduler_type=config["lr_scheduler_type"],
        warmup_ratio=config["warmup_ratio"],
        weight_decay=config["weight_decay"],
        optim=config["optim"],
        max_grad_norm=config["max_grad_norm"],
        num_train_epochs=num_train_epochs if num_train_epochs is not None else config["num_train_epochs"],
        max_steps=max_steps if max_steps is not None else -1,
        eval_strategy=config["eval_strategy"],
        eval_steps=eval_steps if eval_steps is not None else config["eval_steps"],
        save_strategy=config["save_strategy"],
        save_steps=eval_steps if eval_steps is not None else config["save_steps"],
        logging_steps=logging_steps if logging_steps is not None else config["logging_steps"],
        save_total_limit=config["save_total_limit"],
        load_best_model_at_end=config["load_best_model_at_end"],
        metric_for_best_model=config["metric_for_best_model"],
        greater_is_better=config["greater_is_better"],
        max_seq_length=config["max_seq_len"],
        seed=config["seed"],
        bf16=True,
        report_to=[],
        push_to_hub=False,  # pushed manually later, after training finishes and the best checkpoint is loaded
    )
    return SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=raw_datasets["train"],
        eval_dataset=raw_datasets["val"],
        data_collator=collator,
        processing_class=tokenizer,
    )

## 10. Smoke test (~100 steps)

Confirms the loss is actually falling and the model produces valid JSON **before** committing
to the full multi-epoch run. Uses `smoke_test.*` from `train/config.yaml`.

In [ ]:
smoke_output_dir = f"{REPO_ROOT}/{config['smoke_test_output_dir']}"
smoke_trainer = build_trainer(
    output_dir=smoke_output_dir,
    max_steps=config["smoke_test"]["max_steps"],
    eval_steps=config["smoke_test"]["eval_steps"],
    logging_steps=config["smoke_test"]["logging_steps"],
)
smoke_result = smoke_trainer.train()
print(smoke_result)

### Smoke-test loss curve

In [ ]:
import matplotlib.pyplot as plt

history = smoke_trainer.state.log_history
train_pts = [(h["step"], h["loss"]) for h in history if "loss" in h]
eval_pts = [(h["step"], h["eval_loss"]) for h in history if "eval_loss" in h]

plt.figure(figsize=(8, 5))
if train_pts:
    xs, ys = zip(*train_pts)
    plt.plot(xs, ys, label="train loss", marker="o")
if eval_pts:
    xs, ys = zip(*eval_pts)
    plt.plot(xs, ys, label="eval loss", marker="s")
plt.xlabel("step")
plt.ylabel("loss")
plt.title("Smoke test loss (should trend downward)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### 5 generated outputs on val examples (smoke-test model)

In [ ]:
import json
from ticket_router.schema import TicketLabels
from pydantic import ValidationError

def generate_one(model, messages, max_new_tokens=64):
    prompt_ids = tokenizer.apply_chat_template(
        messages[:2], tokenize=True, add_generation_prompt=True, return_tensors="pt",
    ).to(model.device)
    with torch.no_grad():
        out_ids = model.generate(
            prompt_ids, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=tokenizer.pad_token_id,
        )
    completion_ids = out_ids[0][prompt_ids.shape[1]:]
    return tokenizer.decode(completion_ids, skip_special_tokens=True).strip()

n_show = config["smoke_test"]["n_generated_examples"]
for i in range(n_show):
    example = raw_datasets["val"][i]
    generated = generate_one(smoke_trainer.model, example["messages"])
    gold = example["messages"][2]["content"]
    try:
        TicketLabels(**json.loads(generated))
        status = "VALID JSON, matches schema"
    except (json.JSONDecodeError, ValidationError) as e:
        status = f"INVALID: {e}"
    print(f"--- val[{i}] ---")
    print(f"  generated: {generated!r}")
    print(f"  gold:      {gold!r}")
    print(f"  status:    {status}\n")

## 11. STOP AND CONFIRM before continuing

**Do not run the cells below until you've checked, above:**
- The smoke-test loss curve is trending **downward** (not flat, not increasing).
- At least most of the 5 generated outputs are **valid JSON matching the 3-key schema**
  (`queue`, `priority`, `type`) -- they do not need to be *correct* yet at only ~100 steps,
  just structurally valid.

If either looks wrong, stop here and investigate (check `train/config.yaml`, the masking
cell's output, and the collator/model setup above) before spending a full ~45-90 minute run
on it. Everything below this cell is the **full, multi-epoch training run**.

## 12. Full training run

In [ ]:
full_output_dir = f"{REPO_ROOT}/{config['output_dir']}"
trainer = build_trainer(output_dir=full_output_dir)
train_result = trainer.train()
print(train_result)

### Full-run loss curve

In [ ]:
history = trainer.state.log_history
train_pts = [(h["step"], h["loss"]) for h in history if "loss" in h]
eval_pts = [(h["step"], h["eval_loss"]) for h in history if "eval_loss" in h]

plt.figure(figsize=(8, 5))
if train_pts:
    xs, ys = zip(*train_pts)
    plt.plot(xs, ys, label="train loss")
if eval_pts:
    xs, ys = zip(*eval_pts)
    plt.plot(xs, ys, label="eval loss", marker="s")
plt.xlabel("step")
plt.ylabel("loss")
plt.title("Full run: train vs. validation loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

best_ckpt = trainer.state.best_model_checkpoint
best_metric = trainer.state.best_metric
print(f"Best checkpoint (by {config['metric_for_best_model']}): {best_ckpt}")
print(f"Best {config['metric_for_best_model']}: {best_metric}")
print("trainer.train() above was called with load_best_model_at_end=True, so `trainer.model` "
      "is already the BEST checkpoint by validation loss, not simply the last step.")

## 13. Save the adapter + push to the Hugging Face Hub

`trainer.model` is the best checkpoint (see cell 12). Requires `hub_model_id` to be filled in
in `train/config.yaml` -- this notebook will not guess a Hub repo name for you.

In [ ]:
adapter_dir = f"{REPO_ROOT}/{config['adapter_output_dir']}"
trainer.save_model(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f"Adapter + tokenizer saved to {adapter_dir}")

if config["push_to_hub"]:
    if not config["hub_model_id"]:
        raise ValueError(
            "push_to_hub is true in train/config.yaml but hub_model_id is empty. "
            "Set hub_model_id to '<your-hf-username>/<repo-name>' and re-run this cell."
        )
    from huggingface_hub import login
    login()  # paste a Hugging Face access token with write access when prompted
    trainer.model.push_to_hub(config["hub_model_id"], private=config["hub_private"])
    tokenizer.push_to_hub(config["hub_model_id"], private=config["hub_private"])
    print(f"Pushed adapter to https://huggingface.co/{config['hub_model_id']}")
else:
    print("push_to_hub is false in train/config.yaml -- adapter saved locally only.")

## 14. Post-train preview: 20 val examples

A quick eyeball check of the trained model's output format and rough quality, right after
training and before running the project's full evaluation harness (`evaluate.py`) separately.
Uses `post_train_preview.n_val_examples` from `train/config.yaml`.

In [ ]:
n_preview = config["post_train_preview"]["n_val_examples"]
preview_rows = []
for i in range(n_preview):
    example = raw_datasets["val"][i]
    generated = generate_one(trainer.model, example["messages"])
    gold = example["messages"][2]["content"]
    try:
        parsed = TicketLabels(**json.loads(generated))
        status = "valid"
    except (json.JSONDecodeError, ValidationError) as e:
        parsed, status = None, f"invalid: {e}"
    preview_rows.append((i, generated, gold, status))
    print(f"--- val[{i}] ({status}) ---")
    print(f"  generated: {generated!r}")
    print(f"  gold:      {gold!r}\n")

n_valid = sum(1 for _, _, _, s in preview_rows if s == "valid")
n_exact = sum(1 for _, g, gold, s in preview_rows if s == "valid" and g == gold)
print(f"{n_valid}/{n_preview} valid JSON, {n_exact}/{n_preview} exact string match on these {n_preview} examples.")
print("This is a quick eyeball check only -- run src/ticket_router/evaluate.py locally against "
      "a full predictions file for real macro-F1/accuracy numbers on the harness.")